# 01 - Priprema podataka

Ova sveska prati skriptu `scripts/01_prepare_data.py`. Cilj je da se originalni UCI SMS Spam Collection fajl ucita, kratko analizira, ocisti od duplikata i podeli na trening, validacioni i test skup.

Najvaznije pravilo: podela mora biti stratifikovana, da odnos `ham` i `spam` poruka ostane slican u svim skupovima.

In [ ]:
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

ROOT

## Ucitavanje originalnog skupa

Originalni fajl nema klasicno CSV zaglavlje. Svaki red ima labelu, tab karakter i tekst poruke.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

RAW_PATH = ROOT / "data" / "raw" / "SMSSpamCollection"
CLEAN_PATH = ROOT / "data" / "processed" / "sms_spam_clean.csv"
MODELING_PATH = ROOT / "data" / "processed" / "sms_spam_modeling.csv"
SPLIT_DIR = ROOT / "data" / "processed" / "splits"

rows = []
for line in RAW_PATH.read_text(encoding="utf-8", errors="replace").splitlines():
    label, message = line.split("\t", 1)
    rows.append([label, message])

df = pd.DataFrame(rows, columns=["label", "message"])
df.head()

## Osnovna analiza

Dodajemo nekoliko jednostavnih kolona koje pomazu da razumemo strukturu poruka: duzina poruke, broj reci, broj cifara i slicno.

In [ ]:
df["label_id"] = (df["label"] == "spam").astype(int)
df["char_count"] = df["message"].str.len()
df["word_count"] = df["message"].str.split().str.len()
df["digit_count"] = df["message"].str.count(r"\d")
df["has_currency"] = df["message"].str.contains(r"[$\u00a3\u20ac]", regex=True)
df["has_long_number"] = df["message"].str.contains(r"\d{5,}", regex=True)
df["normalized_message"] = df["message"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

class_counts = df["label"].value_counts()
class_percentages = (df["label"].value_counts(normalize=True) * 100).round(2)
duplicate_count = df.duplicated(["label", "message"]).sum()

summary = pd.DataFrame({"broj_poruka": class_counts, "procenat": class_percentages})
display(summary)
print(f"Broj duplikata: {duplicate_count}")

df.groupby("label")[["char_count", "word_count", "digit_count"]].mean().round(2)

## Uklanjanje duplikata

Duplikate uklanjamo pre podele podataka. Tako ista poruka ne moze da se pojavi i u treningu i u testu.

In [ ]:
df_modeling = df.drop_duplicates(["label", "message"]).reset_index(drop=True)

print(f"Broj poruka pre uklanjanja duplikata: {len(df)}")
print(f"Broj poruka posle uklanjanja duplikata: {len(df_modeling)}")
print(f"Uklonjeno duplikata: {len(df) - len(df_modeling)}")

## Stratifikovana podela

Koristimo odnos 70/15/15. Prvo odvajamo 70% za trening, zatim preostalih 30% delimo na validaciju i test.

In [ ]:
train_df, temp_df = train_test_split(
    df_modeling,
    test_size=0.30,
    stratify=df_modeling["label"],
    random_state=42,
)

validation_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42,
)

split_summary = pd.DataFrame({
    "skup": ["train", "validation", "test"],
    "broj_poruka": [len(train_df), len(validation_df), len(test_df)],
    "ham": [sum(train_df["label"] == "ham"), sum(validation_df["label"] == "ham"), sum(test_df["label"] == "ham")],
    "spam": [sum(train_df["label"] == "spam"), sum(validation_df["label"] == "spam"), sum(test_df["label"] == "spam")],
})

split_summary

## Cuvanje pripremljenih podataka

Ove CSV fajlove zatim koriste ostale sveske i Python skripte.

In [ ]:
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(CLEAN_PATH, index=False)
df_modeling.to_csv(MODELING_PATH, index=False)
train_df.to_csv(SPLIT_DIR / "train.csv", index=False)
validation_df.to_csv(SPLIT_DIR / "validation.csv", index=False)
test_df.to_csv(SPLIT_DIR / "test.csv", index=False)

print(f"Sacuvano: {CLEAN_PATH}")
print(f"Sacuvano: {MODELING_PATH}")
print(f"Sacuvani splitovi: {SPLIT_DIR}")